# 01 Feature engineering and training
Ingests every raw extract into its own `src_<source>` schema, builds the canonical project model with provenance,
data-quality checks, the dependency graph, features, and the priority-tier model. Thresholds, mappings, and
rules come from `Files/config`, not from this notebook.

In [ ]:
try:
    import lightgbm  # noqa: F401
except ImportError:
    import subprocess
    import sys

    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "lightgbm"], check=False)

In [ ]:
import ast
import json
import math
import re
import uuid
import datetime as dt
from pathlib import Path
import xml.etree.ElementTree as ET

import numpy as np
import pandas as pd
from pyspark.sql.types import StructType, StructField, StringType

LAKE = Path("/lakehouse/default")
CFG_DIR = LAKE / "Files" / "config"
CONFIG = json.loads((CFG_DIR / "ml-model.json").read_text(encoding="utf-8"))
SOURCES = json.loads((CFG_DIR / "data-sources.json").read_text(encoding="utf-8"))
PORTFOLIO_CFG = json.loads((CFG_DIR / "portfolios.json").read_text(encoding="utf-8"))
RAW = LAKE / SOURCES["lakehouseRawRoot"]
EXPORTS = LAKE / "Files" / "exports"
MODELS = LAKE / "Files" / "models"
EXPORTS.mkdir(parents=True, exist_ok=True)
MODELS.mkdir(parents=True, exist_ok=True)
AS_OF = pd.Timestamp(CONFIG["asOfDate"])
RUN_ID = str(uuid.uuid4())
TIERS = CONFIG["tiers"]["order"]
MAP = CONFIG["mappings"]

In [ ]:
for src in SOURCES["sources"]:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {src['lakehouseSchema']}")
for schema in ("silver", "gold"):
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {schema}")


def clean_name(name):
    return re.sub(r"[^0-9a-zA-Z]+", "_", str(name)).strip("_").lower()


def parse_msproject(path):
    ns = {"p": "http://schemas.microsoft.com/project"}
    tags = ["UID", "ID", "Name", "Summary", "OutlineLevel", "Start", "Finish", "Duration", "PercentComplete", "Work", "Milestone", "Text1"]
    rows = []
    for task in ET.parse(path).getroot().findall(".//p:Task", ns):
        row = {tag: task.findtext(f"p:{tag}", default=None, namespaces=ns) for tag in tags}
        row["PredecessorUID"] = ";".join(link.findtext("p:PredecessorUID", default="", namespaces=ns) for link in task.findall("p:PredecessorLink", ns))
        rows.append(row)
    return pd.DataFrame(rows)


def read_extract(rel_path, table):
    path = RAW / rel_path
    suffix = path.suffix.lower()
    if suffix == ".csv":
        df = pd.read_csv(path, dtype=str, keep_default_na=False)
    elif suffix == ".xlsx":
        df = pd.read_excel(path, sheet_name=table.get("sheet", 0), header=table.get("headerRow", 1) - 1, dtype=str)
    elif suffix == ".json":
        payload = json.loads(path.read_text(encoding="utf-8"))
        if isinstance(payload, dict) and "value" in payload:
            records = payload["value"]
        elif isinstance(payload, dict) and "issues" in payload:
            records = payload["issues"]
        else:
            records = payload
        df = pd.json_normalize(records)
    elif suffix == ".xml":
        df = parse_msproject(path)
    else:
        raise ValueError(f"Unsupported extract: {rel_path}")
    df["source_file"] = path.name
    return df


def is_blank(value):
    if value is None:
        return True
    if isinstance(value, str):
        return value.strip() == ""
    try:
        return bool(pd.isna(value))
    except (TypeError, ValueError):
        return False


def stringify(df):
    out = pd.DataFrame(index=df.index)
    for column in df.columns:
        out[clean_name(column)] = df[column].map(lambda v: None if is_blank(v) else str(v))
    return out


INGESTED = {}
for src in SOURCES["sources"]:
    for table in src["tables"]:
        frames = [read_extract(rel, table) for rel in table["files"]]
        pdf = stringify(pd.concat(frames, ignore_index=True))
        name = f"{src['lakehouseSchema']}.{table['name']}"
        schema = StructType([StructField(c, StringType(), True) for c in pdf.columns])
        spark.createDataFrame(pdf.astype(object).where(pd.notnull(pdf), None), schema=schema).write.mode("overwrite").saveAsTable(name)
        INGESTED[name] = pdf
        print(f"{name}: {len(pdf)} rows, {len(pdf.columns)} columns")

## Normalization helpers
Codes, names, dates, money, yes/no, and risk values are normalized here. Mappings come from `ml-model.json`.

In [ ]:
from pyspark.sql.types import BooleanType, DoubleType, LongType, TimestampType


def parse_fiscal(value):
    if is_blank(value):
        return None
    match = re.search(r"FY\s*(\d{2})\s*Q([1-4])", str(value).upper())
    if not match:
        return None
    fiscal_year, quarter = 2000 + int(match.group(1)), int(match.group(2))
    start = pd.Timestamp(year=fiscal_year - 1, month=7, day=1) + pd.DateOffset(months=3 * (quarter - 1))
    return (start + pd.DateOffset(months=3) - pd.Timedelta(days=1)).normalize()


def norm_code(value):
    if is_blank(value):
        return None
    compact = re.sub(r"[\s_]", "", str(value).strip().upper())
    match = re.fullmatch(r"GIS-?(\d{4})", compact)
    return f"GIS-{match.group(1)}" if match else None


def norm_person(value):
    if is_blank(value):
        return None
    text = " ".join(str(value).split())
    if "," in text:
        last, first = [part.strip() for part in text.split(",", 1)]
        text = f"{first} {last}"
    return text


def parse_date(value):
    if is_blank(value):
        return None
    text = str(value).strip()
    for fmt in ("%m/%d/%Y", "%Y-%m-%d", "%d-%b-%y", "%m/%d/%Y %H:%M", "%Y-%m-%dT%H:%M:%SZ", "%Y-%m-%dT%H:%M:%S"):
        try:
            return pd.Timestamp(dt.datetime.strptime(text, fmt)).normalize()
        except ValueError:
            continue
    return parse_fiscal(text)


def parse_number(value):
    if is_blank(value):
        return None
    try:
        return float(str(value).strip().replace(",", ""))
    except ValueError:
        return None


def parse_money(value):
    if is_blank(value):
        return None
    text = str(value).strip().upper().replace("$", "").replace(",", "")
    scale = 1.0
    if text.endswith("M"):
        scale, text = 1e6, text[:-1]
    elif text.endswith("K"):
        scale, text = 1e3, text[:-1]
    number = parse_number(text)
    return None if number is None else number * scale


def yes_no(value):
    if is_blank(value):
        return None
    return MAP["yesNo"].get(str(value).strip().upper())


def risk_level(value):
    if is_blank(value):
        return None
    text = str(value).strip().lower()
    for name, code in MAP["riskRating"].items():
        if name.lower() == text or (len(text) >= 2 and name.lower().startswith(text)):
            return code
    return MAP["ragToRisk"].get(str(value).strip().title())


def business_value(value):
    if is_blank(value):
        return None
    text = str(value).strip()
    lead = re.match(r"(\d)", text)
    if lead:
        return int(lead.group(1))
    return MAP["businessValueText"].get(text.title())


def sponsor_code(level_name):
    return MAP["sponsorLevel"].get(level_name) if level_name else None


def sponsor_code_from_powerapps(value):
    return sponsor_code(MAP["powerAppsSponsorLevel"].get(str(value).strip())) if not is_blank(value) else None


def sponsor_code_from_excel(value):
    return sponsor_code(MAP["excelSponsorLevel"].get(str(value).strip())) if not is_blank(value) else None


def sponsor_code_from_roster(job_level):
    if is_blank(job_level):
        return None
    return MAP["rosterJobLevelSponsor"].get(str(job_level).strip(), MAP["defaultSponsorLevel"])


ROSTER = INGESTED["src_hr_roster.employees"].copy()
KNOWN_BU = {}
for value in pd.concat([ROSTER["orgunit"], INGESTED["src_powerapps.intake_requests"]["cr5f2_businessunit"], INGESTED["src_excel_intake.intake_10_signals"]["business_unit"]]).dropna():
    KNOWN_BU.setdefault(str(value).strip().lower(), str(value).strip())


def canon_bu(value):
    if is_blank(value):
        return None
    return KNOWN_BU.get(str(value).strip().lower(), str(value).strip())


ROSTER_BY_NAME = {str(r["displayname"]).strip().lower(): r for r in ROSTER.to_dict("records") if not is_blank(r["displayname"])}
PORTFOLIOS = PORTFOLIO_CFG["portfolios"]
PORTFOLIO_BY_NAME = {p["displayName"]: p for p in PORTFOLIOS}
PORTFOLIO_BY_KEY = {p["key"]: p for p in PORTFOLIOS}
PORTFOLIO_BY_CODE = {p["code"]: p for p in PORTFOLIOS}
PROGRAM_CODE = {(p["displayName"], prog["displayName"]): prog["code"] for p in PORTFOLIOS for prog in p["programs"]}
JIRA_PREFIX_TO_KEY = {p["jiraProjectKey"]: p["key"] for p in PORTFOLIOS}
MSPROJECT_FILE_TO_KEY = {Path(p["msProjectFile"]).name: p["key"] for p in PORTFOLIOS}
SKILL_NAMES = MAP["skillNames"]
SKILL_CODE_BY_NAME = {name.lower(): code for code, name in SKILL_NAMES.items()}


def skill_code_from_name(name):
    if is_blank(name):
        return None
    text = str(name).strip().lower()
    if text in SKILL_CODE_BY_NAME:
        return SKILL_CODE_BY_NAME[text]
    matches = [code for code, label in SKILL_NAMES.items() if label.lower().startswith(text) or text.startswith(label.lower())]
    return matches[0] if matches else None


def write_table(pdf, name):
    columns, data = [], {}
    for column in pdf.columns:
        series = pdf[column]
        if series.dtype.kind == "b":
            kind, values = BooleanType(), [None if is_blank(v) else bool(v) for v in series]
        elif series.dtype.kind in "iu":
            kind, values = LongType(), [None if is_blank(v) else int(v) for v in series]
        elif series.dtype.kind == "f":
            kind, values = DoubleType(), [None if is_blank(v) else float(v) for v in series]
        elif series.dtype.kind == "M":
            kind, values = TimestampType(), [None if is_blank(v) else pd.Timestamp(v).to_pydatetime() for v in series]
        else:
            kind, values = StringType(), [None if is_blank(v) else str(v) for v in series]
        columns.append(StructField(column, kind, True))
        data[column] = values
    rows = list(zip(*[data[c] for c in pdf.columns]))
    spark.createDataFrame(rows, schema=StructType(columns)).write.mode("overwrite").saveAsTable(name)
    print(f"{name}: {len(pdf)} rows")
    return len(pdf)

## Per-source views
Each source is reduced to one row per canonical project ID. Multi-row sources are aggregated first.

In [ ]:
def cell(source, pid, column=None):
    if source is None or pid not in source.index:
        return None
    value = source.loc[pid] if column is None else source.at[pid, column]
    return None if is_blank(value) else value


PV_RAW = INGESTED["src_planview.portfolio_items"].copy()
PV_RAW["pid"] = PV_RAW["projectcode"].map(norm_code)
PV = PV_RAW[PV_RAW["pid"].notna()].copy()
PV["go_live"] = PV["plannedgolive"].map(parse_date)
PV["value"] = PV["benefitscore"].map(business_value)
PV["cost"] = PV["estimatedcostusd"].map(parse_money)
PV["risk"] = PV["risklevel"].map(risk_level)
PV["deps"] = PV["dependencycount"].map(parse_number)
PV["protected"] = PV["protectedprogram"].map(yes_no)
PV["sponsor"] = PV["executivesponsor"].map(norm_person)
PV["sponsor_email"] = PV["executivesponsoremail"].map(lambda v: None if is_blank(v) else str(v).strip().lower())
PV["bu"] = PV["businessunit"].map(canon_bu)
PV = PV.drop_duplicates("pid", keep="first").set_index("pid")

PA_RAW = INGESTED["src_powerapps.intake_requests"].copy()
PA_RAW["pid"] = PA_RAW["cr5f2_projectcode"].map(norm_code)
PA = PA_RAW[PA_RAW["pid"].notna()].sort_values("cr5f2_submittedon").drop_duplicates("pid", keep="last").set_index("pid")
PA["go_live"] = PA["cr5f2_targetgolive"].map(parse_date)
PA["value"] = PA["cr5f2_businessvalue"].map(business_value)
PA["sponsor_code"] = PA["cr5f2_sponsorlevel"].map(sponsor_code_from_powerapps)
PA["reg"] = PA["cr5f2_regulatorycompliance"].map(yes_no)
PA["dt"] = PA["cr5f2_dtinitiative"].map(yes_no)
PA["cost"] = PA["cr5f2_estimatedcostk"].map(lambda v: None if parse_number(v) is None else parse_number(v) * 1000)
PA["bu"] = PA["cr5f2_businessunit"].map(canon_bu)
PA["pm"] = PA["cr5f2_requestedby"].map(norm_person)
PA["sponsor_email"] = PA["cr5f2_sponsoremail"].map(lambda v: None if is_blank(v) else str(v).strip().lower())

ADO_RAW = INGESTED["src_ado.work_items"].copy()
ADO_RAW["pid"] = ADO_RAW["custom_projectcode"].map(norm_code)
ADO = ADO_RAW[ADO_RAW["pid"].notna()].copy()
ADO["points"] = ADO["story_points"].map(parse_number)
ADO["dt_tag"] = ADO["tags"].fillna("").str.contains("dt-initiative")


def portfolio_key_from_area(value):
    if is_blank(value):
        return None
    parts = str(value).split("\\")
    portfolio = PORTFOLIO_BY_CODE.get(parts[1]) if len(parts) > 1 else None
    return portfolio["key"] if portfolio else None


ADO["portfolio_key"] = ADO["area_path"].map(portfolio_key_from_area)
ado_agg = ADO.groupby("pid").agg(points=("points", "sum"), dt=("dt_tag", "any"), rows=("points", "size"))
ado_portfolio = ADO.dropna(subset=["portfolio_key"]).groupby("pid")["portfolio_key"].first()

JIRA_RAW = json.loads((RAW / "jira" / "jira_issues_export.json").read_text(encoding="utf-8"))["issues"]
KEY_TO_PID = {issue["key"]: norm_code((issue.get("fields") or {}).get("customfield_10050")) for issue in JIRA_RAW}
JIRA = INGESTED["src_jira.issues"].copy()
JIRA["pid"] = JIRA["fields_customfield_10050"].map(norm_code)
JIRA["points"] = JIRA["fields_customfield_10016"].map(parse_number)
JIRA["dt_tag"] = JIRA["fields_labels"].fillna("").str.contains("dt-initiative")
JIRA["portfolio_key"] = JIRA["key"].map(lambda k: JIRA_PREFIX_TO_KEY.get(str(k).split("-")[0]))
JIRA = JIRA[JIRA["pid"].notna()]
jira_agg = JIRA.groupby("pid").agg(points=("points", "sum"), dt=("dt_tag", "any"), portfolio_key=("portfolio_key", "first"), rows=("points", "size"))

MSP_RAW = INGESTED["src_ms_project.tasks"].copy()
MSP_RAW["pid"] = MSP_RAW["text1"].map(norm_code)
MSP = MSP_RAW[MSP_RAW["pid"].notna()].copy()
MSP["hours"] = MSP["work"].map(lambda v: None if is_blank(v) or not re.search(r"PT(\d+)H", str(v)) else float(re.search(r"PT(\d+)H", str(v)).group(1)))
msp_summary = MSP[MSP["summary"] == "1"].drop_duplicates("pid").set_index("pid")
msp_hours = MSP[MSP["summary"] == "0"].groupby("pid")["hours"].sum()
msp_file = MSP.drop_duplicates("pid").set_index("pid")["source_file"]

SS_RAW = INGESTED["src_smartsheet.milestones"].copy()
SS_RAW["pid"] = SS_RAW["project_code"].map(norm_code)
SS = SS_RAW[SS_RAW["pid"].notna()].copy()
ss_golive = SS[SS["milestone_task"] == "Go-Live"].drop_duplicates("pid").set_index("pid")
ss_golive = ss_golive.assign(golive=ss_golive["finish"].map(parse_date))
ss_rag = SS[SS["rag"].notna()].drop_duplicates("pid").set_index("pid")["rag"].map(risk_level)

TR_RAW = INGESTED["src_spreadsheets.bu_tracker"].copy()
TR_RAW["pid"] = TR_RAW["proj"].map(norm_code)
TR = TR_RAW[TR_RAW["pid"].notna()].copy()
TR["fy_end"] = TR["target_live_fy_qtr"].map(parse_fiscal)
TR["budget"] = TR["budget_k"].map(lambda v: None if parse_number(v) is None else parse_number(v) * 1000)
TR["reg"] = TR["regulatory_y_n"].map(yes_no)
TR = TR.drop_duplicates("pid", keep="first").set_index("pid")

DEC_RAW = INGESTED["src_spreadsheets.pmo_decisions"].copy()
DEC_RAW["pid"] = DEC_RAW["project_code"].map(norm_code)
DEC = DEC_RAW[DEC_RAW["pid"].notna()].copy()

REPO_DOCS = json.loads((RAW / "bu_repositories" / "bu_document_index.json").read_text(encoding="utf-8"))
REPO_BY_PID = {}
for doc in REPO_DOCS:
    pid = norm_code(doc.get("projectRef"))
    if pid:
        REPO_BY_PID.setdefault(pid, []).append(doc)

ALLOC_RAW = INGESTED["src_hr_capacity.allocations"].copy()
ALLOC_RAW["pid"] = ALLOC_RAW["projcode"].map(norm_code)
ALLOC_RAW["pct"] = ALLOC_RAW["allocationpct"].map(parse_number)
ALLOC_MONTHS = CONFIG["capacity"]["allocationMonths"]
ALLOC = ALLOC_RAW[ALLOC_RAW["pid"].notna()].copy()
alloc_recent = ALLOC[ALLOC["monthstart"].isin(ALLOC_MONTHS)]
alloc_fte = alloc_recent.groupby(["pid", "skillcode"])["pct"].sum().div(100 * len(ALLOC_MONTHS)).rename("fte").reset_index()
alloc_primary = alloc_fte.sort_values("fte", ascending=False).drop_duplicates("pid").set_index("pid")

COST_RAW = INGESTED["src_cost.project_cost_monthly"].copy()
COST_RAW["pid"] = COST_RAW["projectcode"].map(norm_code)
COST = COST_RAW[COST_RAW["pid"].notna()].copy()
COST["budget"] = COST["budgetusd"].map(parse_number)
cost_annual = COST.groupby("pid")["budget"].sum()

XL_RAW = INGESTED["src_excel_intake.intake_10_signals"].copy()
SIGNAL_COLS = list(XL_RAW.columns[6:16])
XL_RAW["pid"] = XL_RAW["project_id"].map(norm_code)
XL = XL_RAW[XL_RAW["pid"].notna()].drop_duplicates("pid").set_index("pid")
skill_parts = XL[SIGNAL_COLS[4]].map(lambda v: None if is_blank(v) else [part.strip() for part in str(v).split(":", 1)])
XLD = pd.DataFrame(index=XL.index)
XLD["golive"] = XL[SIGNAL_COLS[0]].map(parse_date)
XLD["value"] = XL[SIGNAL_COLS[1]].map(business_value)
XLD["sponsor_code"] = XL[SIGNAL_COLS[2]].map(sponsor_code_from_excel)
XLD["reg"] = XL[SIGNAL_COLS[3]].map(yes_no)
XLD["skill"] = skill_parts.map(lambda parts: None if parts is None or len(parts) < 2 else skill_code_from_name(parts[0]))
XLD["fte"] = skill_parts.map(lambda parts: None if parts is None or len(parts) < 2 else parse_number(parts[1]))
XLD["cost"] = XL[SIGNAL_COLS[5]].map(parse_money)
XLD["deps"] = XL[SIGNAL_COLS[6]].map(parse_number)
XLD["risk"] = XL[SIGNAL_COLS[7]].map(risk_level)
XLD["dt"] = XL[SIGNAL_COLS[8]].map(yes_no)
XLD["effort"] = XL[SIGNAL_COLS[9]].map(parse_number)
XLD["bu"] = XL["business_unit"].map(canon_bu)
XLD["site"] = XL["site"]
XLD["portfolio"] = XL["portfolio"]
XLD["name"] = XL["project_name"]
XLD["sponsor"] = XL["sponsor_name"].map(norm_person)

In [ ]:
EDGES = []
for doc in REPO_DOCS:
    source_pid = norm_code(doc.get("projectRef"))
    if source_pid is None or doc.get("docType") != "DependencyRegister":
        continue
    for dependency in doc.get("dependencies") or []:
        target = norm_code(dependency.get("dependsOnProject"))
        EDGES.append((source_pid, target, dependency.get("linkType") or "FinishToStart", "bu_repositories"))
for issue in JIRA_RAW:
    source_pid = KEY_TO_PID.get(issue["key"])
    for link in (issue.get("fields") or {}).get("issuelinks") or []:
        if (link.get("type") or {}).get("name") == "Blocks":
            target = KEY_TO_PID.get((link.get("outwardIssue") or {}).get("key"))
            if source_pid and target:
                EDGES.append((source_pid, target, "Blocks", "jira"))
for pid, row in TR.iterrows():
    for code in str(row.get("depends_on_codes") or "").split(","):
        target = norm_code(code)
        if target:
            EDGES.append((pid, target, "FinishToStart", "bu_trackers"))
edge_df = pd.DataFrame(EDGES, columns=["from_id", "to_id", "relationship", "source"]).dropna()
edge_df = edge_df[edge_df["from_id"] != edge_df["to_id"]]
edge_table = edge_df.groupby(["from_id", "to_id"]).agg(
    relationship_type=("relationship", "first"),
    evidence_source=("source", lambda s: ",".join(sorted(set(s)))),
    evidence_count=("source", "size"),
).reset_index()
out_degree = edge_table.groupby("from_id")["to_id"].nunique()

SS_PIDS = set(SS["pid"])
ALLOC_PIDS = set(ALLOC["pid"])
DEC_PIDS = set(DEC["pid"])
ALL_IDS = sorted(set(PV.index) | set(PA.index) | set(ado_agg.index) | set(jira_agg.index) | set(msp_summary.index) | set(SS_PIDS) | set(TR.index) | set(cost_annual.index) | set(XLD.index) | set(alloc_primary.index) | set(DEC_PIDS) | set(REPO_BY_PID.keys()))
print(f"canonical project IDs: {len(ALL_IDS)}")
assert len(ALL_IDS) == 100, f"expected 100 projects, found {len(ALL_IDS)}"

## Resolve the ten signals with provenance
The first non-blank value in precedence order wins. Every choice is written to `silver.signal_provenance`.

In [ ]:
def resolve(signal, candidates, provenance, pid):
    for source, value in candidates:
        if not is_blank(value):
            provenance.append((pid, signal, source))
            return value
    provenance.append((pid, signal, None))
    return None


def repo_risk(docs):
    severities = [MAP["riskRating"].get(item.get("severity")) for doc in docs for item in (doc.get("riskItems") or []) if item.get("status") == "Open"]
    severities = [s for s in severities if s]
    return max(severities) if severities else None


def repo_regulatory(docs):
    if not docs:
        return None
    return 1 if any(doc.get("complianceTags") for doc in docs) else None


def run_checks(pid, context):
    results = []
    for check in CONFIG["dataQuality"]["checks"]:
        value = context.get(check.get("field"))
        kind, evaluated, passed, detail = check["type"], True, None, ""
        if kind == "dateRange":
            evaluated = value is not None
            passed = evaluated and pd.Timestamp(check["min"]) <= value <= pd.Timestamp(check["max"])
        elif kind == "intRange":
            evaluated = value is not None
            passed = evaluated and check["min"] <= value <= check["max"]
        elif kind == "numberRange":
            evaluated = value is not None
            if evaluated:
                low_ok = value > check["minExclusive"] if "minExclusive" in check else value >= check["min"]
                passed = low_ok and value <= check["max"]
        elif kind == "inList":
            evaluated = value is not None
            passed = evaluated and value in check["values"]
        elif kind == "notNull":
            passed = bool(value)
        elif kind == "codeFormat":
            evaluated = value is not None
            if evaluated:
                passed = bool(re.fullmatch(check["pattern"], str(value))) and (value == pid if check.get("requireCleanInput") else True)
        elif kind == "spreadDays":
            dates = [d for d in (value or []) if d is not None]
            evaluated = len(dates) >= check["minSources"]
            if evaluated:
                spread = (max(dates) - min(dates)).days
                passed, detail = spread <= check["maxDays"], f"spread {spread} days"
        elif kind == "spreadPercent":
            numbers = [n for n in (value or []) if n]
            evaluated = len(numbers) >= check["minSources"]
            if evaluated:
                spread = (max(numbers) - min(numbers)) / min(numbers) * 100
                passed, detail = spread <= check["maxPercent"], f"spread {spread:.1f} percent"
        else:
            evaluated = False
        results.append((check["id"], evaluated, bool(passed) if evaluated else None, detail))
    return results

In [ ]:
SOURCE_NAMES = ["planview", "powerapps", "ado", "jira", "ms_project", "smartsheet", "bu_trackers", "bu_repositories", "hr_capacity", "cost", "excel_intake"]
RECORD_COUNTS = {
    "planview": PV_RAW["pid"].value_counts(),
    "powerapps": PA_RAW["pid"].value_counts(),
    "ado": ADO_RAW["pid"].value_counts(),
    "jira": JIRA["pid"].value_counts(),
    "ms_project": MSP_RAW["pid"].value_counts(),
    "smartsheet": SS_RAW["pid"].value_counts(),
    "bu_trackers": TR_RAW["pid"].value_counts(),
    "bu_repositories": pd.Series([norm_code(d.get("projectRef")) for d in REPO_DOCS]).value_counts(),
    "hr_capacity": ALLOC_RAW["pid"].value_counts(),
    "cost": COST_RAW["pid"].value_counts(),
    "excel_intake": XL_RAW["pid"].value_counts(),
}
SIGNAL_KEYS = ["go_live_date", "business_value", "sponsor_priority", "regulatory_security", "primary_skill", "cost_usd", "dependencies", "risk", "dt_alignment", "effort_person_weeks"]

canonical, provenance, coverage, dq_rows = [], [], [], []
for pid in ALL_IDS:
    portfolio_name = resolve("portfolio", [("planview", cell(PV, pid, "portfolio")), ("excel_intake", cell(XLD, pid, "portfolio"))], provenance, pid)
    portfolio = PORTFOLIO_BY_NAME.get(portfolio_name)
    if portfolio is None:
        key = cell(ado_portfolio, pid) or cell(jira_agg, pid, "portfolio_key") or MSPROJECT_FILE_TO_KEY.get(cell(msp_file, pid) or "")
        if key is None:
            raise ValueError(f"No portfolio evidence for {pid}")
        portfolio = PORTFOLIO_BY_KEY[key]
        provenance.append((pid, "portfolio", "fallback_" + ("ado" if cell(ado_portfolio, pid) else "jira_or_ms_project")))
    program_name = resolve("program", [("planview", cell(PV, pid, "programname"))], provenance, pid)
    program_code = PROGRAM_CODE.get((portfolio["displayName"], program_name))
    if program_code is None:
        program_name, program_code = portfolio["programs"][0]["displayName"], portfolio["programs"][0]["code"]

    name = resolve("project_name", [("planview", cell(PV, pid, "projectname")), ("powerapps", cell(PA, pid, "cr5f2_requesttitle")), ("excel_intake", cell(XLD, pid, "name"))], provenance, pid)
    bu = resolve("business_unit", [("planview", cell(PV, pid, "bu")), ("powerapps", cell(PA, pid, "bu")), ("excel_intake", cell(XLD, pid, "bu"))], provenance, pid)
    site = resolve("site", [("planview", cell(PV, pid, "site")), ("excel_intake", cell(XLD, pid, "site"))], provenance, pid)
    pm = resolve("project_manager", [("planview", norm_person(cell(PV, pid, "projectmanager"))), ("powerapps", cell(PA, pid, "pm"))], provenance, pid)
    sponsor = resolve("executive_sponsor", [("planview", cell(PV, pid, "sponsor")), ("excel_intake", cell(XLD, pid, "sponsor"))], provenance, pid)
    sponsor_email = resolve("sponsor_email", [("planview", cell(PV, pid, "sponsor_email")), ("powerapps", cell(PA, pid, "sponsor_email"))], provenance, pid)
    roster_row = ROSTER_BY_NAME.get(str(sponsor).strip().lower()) if sponsor else None
    sponsor_match = roster_row is not None
    status = cell(PV, pid, "projectstatus") or "Planning"
    protected = resolve("protected_program", [("planview", cell(PV, pid, "protected"))], provenance, pid) or 0

    raw_code = next((code for code in [cell(PV, pid, "projectcode"), cell(PA, pid, "cr5f2_projectcode"), cell(XL, pid, "project_id")] if code), None)
    docs = REPO_BY_PID.get(pid, [])

    golive_sources = [
        ("planview", cell(PV, pid, "go_live")),
        ("smartsheet", cell(ss_golive, pid, "golive")),
        ("powerapps", cell(PA, pid, "go_live")),
        ("excel_intake", cell(XLD, pid, "golive")),
        ("bu_trackers", cell(TR, pid, "fy_end")),
        ("ms_project", parse_date(cell(msp_summary, pid, "finish"))),
    ]
    go_live = resolve("go_live_date", golive_sources, provenance, pid)
    months = None if go_live is None else (go_live - AS_OF).days / 30.44
    value = resolve("business_value", [("planview", cell(PV, pid, "value")), ("powerapps", cell(PA, pid, "value")), ("excel_intake", cell(XLD, pid, "value"))], provenance, pid)
    roster_level = sponsor_code_from_roster(roster_row["joblevel"]) if roster_row else None
    sponsor_priority = resolve("sponsor_priority", [("powerapps", cell(PA, pid, "sponsor_code")), ("hr_roster", roster_level), ("excel_intake", cell(XLD, pid, "sponsor_code"))], provenance, pid)
    regulatory = resolve("regulatory_security", [("powerapps", cell(PA, pid, "reg")), ("excel_intake", cell(XLD, pid, "reg")), ("bu_repositories", repo_regulatory(docs)), ("bu_trackers", cell(TR, pid, "reg"))], provenance, pid)
    pv_strategic = cell(PV, pid, "strategictag")
    tag_dt = 1 if (cell(ado_agg, pid, "dt") or cell(jira_agg, pid, "dt")) else None
    dt_flag = resolve("dt_alignment", [("powerapps", cell(PA, pid, "dt")), ("planview", None if pv_strategic is None else int(pv_strategic == "DT Initiative")), ("ado_jira", tag_dt), ("excel_intake", cell(XLD, pid, "dt"))], provenance, pid)
    skill_code = resolve("primary_skill", [("planview", skill_code_from_name(cell(PV, pid, "primaryskill"))), ("excel_intake", cell(XLD, pid, "skill")), ("hr_capacity", cell(alloc_primary, pid, "skillcode"))], provenance, pid)
    fte = resolve("skill_demand_fte", [("planview", parse_number(cell(PV, pid, "skilldemandfte"))), ("excel_intake", cell(XLD, pid, "fte")), ("hr_capacity", cell(alloc_primary, pid, "fte"))], provenance, pid)
    cost = resolve("cost_usd", [("finance_cost", cell(cost_annual, pid)), ("planview", cell(PV, pid, "cost")), ("powerapps", cell(PA, pid, "cost")), ("bu_trackers", cell(TR, pid, "budget")), ("excel_intake", cell(XLD, pid, "cost"))], provenance, pid)
    graph_deps = int(out_degree.get(pid, 0)) or None
    deps = resolve("dependencies", [("dependency_graph", graph_deps), ("planview", cell(PV, pid, "deps")), ("excel_intake", cell(XLD, pid, "deps"))], provenance, pid)
    risk = resolve("risk", [("planview", cell(PV, pid, "risk")), ("bu_repositories", repo_risk(docs) or None), ("smartsheet", cell(ss_rag, pid)), ("excel_intake", cell(XLD, pid, "risk"))], provenance, pid)
    msp_hours_value = cell(msp_hours, pid)
    story_points = (cell(ado_agg, pid, "points") or 0) + (cell(jira_agg, pid, "points") or 0)
    effort = resolve("effort_person_weeks", [("ms_project", None if msp_hours_value is None else msp_hours_value / 40), ("excel_intake", cell(XLD, pid, "effort")), ("ado_jira", story_points * 0.5 if story_points else None)], provenance, pid)

    signal_values = [go_live, value, sponsor_priority, regulatory, skill_code, cost, deps, risk, dt_flag, effort]
    signals_present = sum(1 for v in signal_values if v is not None)
    golive_candidates = [v for _, v in golive_sources if v is not None]
    cost_candidates = [v for v in [cell(PV, pid, "cost"), cell(PA, pid, "cost"), cell(TR, pid, "budget"), cell(cost_annual, pid), cell(XLD, pid, "cost")] if v is not None]
    context = {
        "go_live_date": go_live, "business_value": value, "cost_usd": cost, "dependencies": deps,
        "effort_person_weeks": effort, "risk": risk, "sponsor_roster_match": sponsor_match,
        "project_id_raw": raw_code, "go_live_candidates": golive_candidates, "cost_candidates": cost_candidates,
    }
    for check_id, evaluated, passed, detail in run_checks(pid, context):
        dq_rows.append((pid, check_id, int(bool(passed)) if evaluated else None, int(evaluated), detail))

    for source in SOURCE_NAMES:
        count = int(RECORD_COUNTS[source].get(pid, 0))
        coverage.append((pid, source, int(count > 0), count))

    canonical.append({
        "project_id": pid,
        "project_name": name,
        "portfolio_key": portfolio["key"],
        "portfolio_name": portfolio["displayName"],
        "portfolio_code": portfolio["code"],
        "portfolio_role": portfolio["role"],
        "program_name": program_name,
        "program_code": program_code,
        "business_unit": bu,
        "site": site,
        "project_manager": pm,
        "executive_sponsor": sponsor,
        "sponsor_email": sponsor_email,
        "sponsor_roster_match": int(sponsor_match),
        "sponsor_level_code": sponsor_priority,
        "project_status": status,
        "protected_program": int(protected),
        "go_live_date": go_live,
        "months_to_go_live": months,
        "business_value": value,
        "sponsor_priority": sponsor_priority,
        "regulatory_security": regulatory,
        "dt_alignment": dt_flag,
        "primary_skill_code": skill_code,
        "primary_skill": SKILL_NAMES.get(skill_code) if skill_code else None,
        "skill_demand_fte": fte,
        "cost_usd": cost,
        "dependencies": deps,
        "risk": risk,
        "effort_person_weeks": effort,
        "signals_present": signals_present,
        "project_id_raw": raw_code,
    })

CANON = pd.DataFrame(canonical)
CANON["sponsor_priority"] = CANON["sponsor_priority"].astype("float")
PROV = pd.DataFrame(provenance, columns=["project_id", "signal", "source"])
COVERAGE = pd.DataFrame(coverage, columns=["project_id", "source_name", "present", "record_count"])
DQ = pd.DataFrame(dq_rows, columns=["project_id", "check_id", "passed", "evaluated", "detail"])
DQ = DQ[DQ["evaluated"] == 1].copy()
DQ["passed"] = DQ["passed"].astype(int)
print(f"signals present: {CANON['signals_present'].value_counts().sort_index().to_dict()}")
print(f"DQ checks evaluated: {len(DQ)}, failed: {int((DQ['passed'] == 0).sum())}")

In [ ]:
PORTFOLIO_CODE = dict(zip(CANON["project_id"], CANON["portfolio_code"]))
DEP = edge_table.copy()
DEP = DEP[DEP["from_id"].isin(PORTFOLIO_CODE) & DEP["to_id"].isin(PORTFOLIO_CODE)].copy()
DEP["from_portfolio_code"] = DEP["from_id"].map(PORTFOLIO_CODE)
DEP["to_portfolio_code"] = DEP["to_id"].map(PORTFOLIO_CODE)
DEP["is_cross_portfolio"] = (DEP["from_portfolio_code"] != DEP["to_portfolio_code"]).astype(int)
DEP = DEP.rename(columns={"from_id": "from_project_id", "to_id": "to_project_id"})[
    ["from_project_id", "to_project_id", "from_portfolio_code", "to_portfolio_code", "is_cross_portfolio", "relationship_type", "evidence_source", "evidence_count"]
].reset_index(drop=True)

write_table(CANON.drop(columns=["sponsor_level_code"]), "silver.project_canonical")
write_table(PROV, "silver.signal_provenance")
write_table(COVERAGE, "gold.source_coverage")
write_table(DQ.drop(columns=["evaluated"]), "gold.dq_checks")
write_table(DEP, "gold.project_dependencies")

## Features, labels, rules, and the priority-tier model
Features use only observed signals. The PMO decision history is the training label. The model is LightGBM with
calibrated probabilities. A multinomial logistic regression is the baseline.

In [ ]:
FEATURE_NAMES = [f["name"] for f in CONFIG["features"]]
TIER_INDEX = {tier: i for i, tier in enumerate(TIERS)}


def urgency(months):
    if is_blank(months):
        return None
    horizon = CONFIG["urgency"]["horizonMonths"]
    return min(1.0, max(0.0, (horizon - months) / horizon))


feat = pd.DataFrame({"project_id": CANON["project_id"]})
feat["timing_urgency"] = CANON["months_to_go_live"].map(urgency).astype(float)
feat["business_value"] = CANON["business_value"].astype(float)
feat["sponsor_priority"] = CANON["sponsor_priority"].astype(float)
feat["regulatory_security"] = CANON["regulatory_security"].astype(float)
feat["skill_demand"] = CANON["skill_demand_fte"].astype(float)
feat["cost_log10"] = CANON["cost_usd"].map(lambda v: None if is_blank(v) else math.log10(float(v))).astype(float)
feat["dependencies"] = CANON["dependencies"].astype(float)
feat["risk"] = CANON["risk"].astype(float)
feat["dt_alignment"] = CANON["dt_alignment"].astype(float)
feat["effort"] = CANON["effort_person_weeks"].astype(float)
dq_summary = DQ.groupby("project_id").agg(passed=("passed", "sum"), evaluated=("passed", "size")).reset_index()
feat = feat.merge(dq_summary, on="project_id", how="left")
feat["quality"] = (feat["passed"] / feat["evaluated"]).fillna(0.0)
feat["completeness"] = CANON["signals_present"].values / len(CONFIG["features"])
weights = CONFIG["model"]["sampleWeight"]
feat["sample_weight"] = weights["base"] + weights["completenessQualityWeight"] * feat["completeness"] * feat["quality"]
feat = feat.drop(columns=["passed", "evaluated"])
write_table(feat, "gold.project_features")

In [ ]:
DEC_LABELS = DEC.assign(review_date=DEC["review_date"].map(parse_date)).sort_values("review_date").drop_duplicates("pid", keep="last")
LABELS = DEC_LABELS.rename(columns={"pid": "project_id"})[["project_id", "review_date", "decision_tier"]]
LABELS = LABELS[LABELS["decision_tier"].isin(TIERS)].reset_index(drop=True)
write_table(LABELS, "gold.pmo_decision_labels")

RULES = pd.DataFrame(
    [(r["id"], r["name"], bool(r["enabled"]), json.dumps(r["when"]), json.dumps(r["then"]), r["owner"]) for r in CONFIG["rules"]],
    columns=["rule_id", "rule_name", "enabled", "when_json", "then_json", "owner"],
)
write_table(RULES, "gold.pmo_rules")

In [ ]:
import lightgbm as lgb
import joblib
from sklearn.calibration import CalibratedClassifierCV
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, log_loss
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

train = feat.merge(LABELS[["project_id", "decision_tier"]], on="project_id", how="inner")
X = train[FEATURE_NAMES].astype(float)
y = train["decision_tier"].map(TIER_INDEX).astype(int).to_numpy()
sample_weight = train["sample_weight"].to_numpy()
class_counts = np.bincount(y, minlength=len(TIERS))
min_class = int(class_counts[class_counts > 0].min())
cv_cfg = CONFIG["model"]["crossValidation"]
folds = max(cv_cfg["minFoldsWhenClassSmall"], min(cv_cfg["folds"], min_class))
params = CONFIG["model"]["classifier"]["params"]
skf = StratifiedKFold(n_splits=folds, shuffle=True, random_state=params["random_state"])


def full_proba(model, rows):
    proba = np.zeros((rows.shape[0], len(TIERS)))
    proba[:, model.classes_] = model.predict_proba(rows)
    return proba


lgb_scores, lr_scores = [], []
for train_idx, test_idx in skf.split(X, y):
    gbm = lgb.LGBMClassifier(**params)
    gbm.fit(X.iloc[train_idx], y[train_idx], sample_weight=sample_weight[train_idx])
    proba = full_proba(gbm, X.iloc[test_idx])
    lgb_scores.append((accuracy_score(y[test_idx], proba.argmax(1)), f1_score(y[test_idx], proba.argmax(1), average="macro", zero_division=0), log_loss(y[test_idx], proba, labels=list(range(len(TIERS))))))
    baseline = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), LogisticRegression(max_iter=CONFIG["model"]["baseline"]["maxIter"], C=1 / CONFIG["model"]["baseline"]["regParam"]))
    baseline.fit(X.iloc[train_idx], y[train_idx], logisticregression__sample_weight=sample_weight[train_idx])
    base_proba = np.zeros((len(test_idx), len(TIERS)))
    base_proba[:, baseline.classes_] = baseline.predict_proba(X.iloc[test_idx])
    lr_scores.append((accuracy_score(y[test_idx], base_proba.argmax(1)), f1_score(y[test_idx], base_proba.argmax(1), average="macro", zero_division=0), log_loss(y[test_idx], base_proba, labels=list(range(len(TIERS))))))

lgb_mean = np.mean(lgb_scores, axis=0)
lr_mean = np.mean(lr_scores, axis=0)
final_gbm = lgb.LGBMClassifier(**params)
final_gbm.fit(X, y, sample_weight=sample_weight)
calibration_cv = max(2, min(CONFIG["model"]["calibration"]["cv"], min_class))
calibrated = CalibratedClassifierCV(estimator=lgb.LGBMClassifier(**params), method=CONFIG["model"]["calibration"]["method"], cv=calibration_cv)
calibrated.fit(X, y, sample_weight=sample_weight)
final_gbm.booster_.save_model(str(MODELS / CONFIG["model"]["classifier"]["boosterFile"]))
joblib.dump(calibrated, MODELS / CONFIG["model"]["calibration"]["calibratedFile"])
(MODELS / "feature_names.json").write_text(json.dumps(FEATURE_NAMES), encoding="utf-8")

now = dt.datetime.now(dt.timezone.utc).replace(tzinfo=None)
metric_rows = []
for model_name, scores in ((CONFIG["mlflow"]["modelName"] + "_lightgbm", lgb_mean), ("logistic_regression_baseline", lr_mean)):
    for metric_name, value in zip(["accuracy_cv", "macro_f1_cv", "log_loss_cv"], scores):
        metric_rows.append((model_name, metric_name, float(value), RUN_ID, now))
metric_rows.append((CONFIG["mlflow"]["modelName"] + "_lightgbm", "training_rows", float(len(train)), RUN_ID, now))
METRICS = pd.DataFrame(metric_rows, columns=["model_name", "metric_name", "metric_value", "run_id", "recorded_utc"])
write_table(METRICS, "gold.model_metrics")

try:
    import mlflow
    mlflow.set_experiment(CONFIG["mlflow"]["experiment"])
    with mlflow.start_run(run_name="priority_tier_training"):
        mlflow.log_metrics({f"{row.model_name}.{row.metric_name}": row.metric_value for row in METRICS.itertuples()})
except Exception as error:
    print(f"MLflow logging skipped: {error}")

In [ ]:
summary = {
    "runId": RUN_ID,
    "projects": int(len(CANON)),
    "sourcesIngested": len(INGESTED),
    "tablesWritten": ["silver.project_canonical", "silver.signal_provenance", "gold.source_coverage", "gold.dq_checks", "gold.project_dependencies", "gold.project_features", "gold.pmo_decision_labels", "gold.pmo_rules", "gold.model_metrics"],
    "crossPortfolioDependencies": int(DEP["is_cross_portfolio"].sum()),
    "dependencyEdges": int(len(DEP)),
    "trainingRows": int(len(train)),
    "classCounts": {TIERS[i]: int(c) for i, c in enumerate(class_counts)},
    "cvFolds": int(folds),
    "lightgbmCv": {"accuracy": float(lgb_mean[0]), "macroF1": float(lgb_mean[1]), "logLoss": float(lgb_mean[2])},
    "baselineCv": {"accuracy": float(lr_mean[0]), "macroF1": float(lr_mean[1]), "logLoss": float(lr_mean[2])},
    "dqFailedChecks": int((DQ["passed"] == 0).sum()),
    "signalsPresentDistribution": {int(k): int(v) for k, v in CANON["signals_present"].value_counts().sort_index().items()},
}
(EXPORTS / "notebook01_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
print(json.dumps(summary, indent=2))